# LeetCode 3678 — Smallest Absent Positive Greater Than Average bằng Python

Link đề: https://leetcode.com/problems/smallest-absent-positive-greater-than-average/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def smallestAbsent(self, nums: List[int]) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm số nguyên **dương nhỏ nhất** thỏa **đồng thời** hai điều kiện: (a) **không có** trong `nums`, (b) **lớn hơn hẳn** giá trị trung bình của `nums`.
2. Tính trung bình **chính xác** bằng số nguyên (so sánh `x * n > tổng`, hoặc điểm bắt đầu `tổng // n + 1`) thay vì số thực, và hiểu khác biệt giữa `int(...)` và `//` với số **âm**.
3. Dùng `set` (hoặc mảng cờ) để kiểm tra "có trong `nums` không" trong `O(1)` rồi **tăng dần ứng viên** cho đến khi tìm được số vắng mặt.
4. Viết 4 cách (thử từng số với `list`, `set`, mảng cờ, quét mảng đã sắp xếp) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: dùng `>=` thay `>`, bỏ qua điều kiện trung bình, cắt phần thập phân bằng `int()` khi trung bình âm, và quên bỏ qua các số **đã có** trong `nums`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trả về **số nguyên dương nhỏ nhất** mà:

- **không xuất hiện** trong `nums`, và
- **lớn hơn hẳn** giá trị **trung bình** của các phần tử `nums` (trung bình = tổng / số phần tử).

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[3,5]` | `6` | trung bình `(3+5)/2 = 4`; số dương nhỏ nhất lớn hơn `4` mà vắng mặt là `6` (vì `5` có mặt) |
| 2 | `[-1,1,2]` | `3` | trung bình `2/3 ~ 0.667`; `1` và `2` có mặt nên đáp án `3` |
| 3 | `[4,-1]` | `2` | trung bình `3/2 = 1.5`; số dương nhỏ nhất lớn hơn `1.5` là `2`, và `2` vắng mặt |

Constraints: `1 <= nums.length <= 100`, `-100 <= nums[i] <= 100`.

In [1]:
nums = [3, 5]
avg = sum(nums) / len(nums)
print("trung bình =", avg)
present = set(nums)
x = 1
while not (x > avg and x not in present):
    x += 1
print("đáp án =", x)
for nums in ([-1, 1, 2], [4, -1]):
    avg = sum(nums) / len(nums)
    x = 1
    while not (x > avg and x not in set(nums)):
        x += 1
    print(nums, "trung bình", round(avg, 3), "->", x)

trung bình = 4.0
đáp án = 6
[-1, 1, 2] trung bình 0.667 -> 3
[4, -1] trung bình 1.5 -> 2


## 3. Kiến thức nền

### 3.1. Trung bình và "lớn hơn hẳn"

Trung bình `avg = sum(nums) / len(nums)` có thể là số thực. "Lớn hơn hẳn" nghĩa là **`>` ngặt**: nếu `avg = 4` thì `4` **không** thỏa, phải là `5` trở lên. Số nguyên nhỏ nhất `> avg` luôn bằng `floor(avg) + 1` (kể cả khi `avg` là số nguyên: `floor(4) + 1 = 5`).

In [2]:
import math
for avg in (4, 1.5, 0.667, -0.5, -2.5):
    print(avg, "-> số nguyên nhỏ nhất > avg =", math.floor(avg) + 1)

4 -> số nguyên nhỏ nhất > avg = 5
1.5 -> số nguyên nhỏ nhất > avg = 2
0.667 -> số nguyên nhỏ nhất > avg = 1
-0.5 -> số nguyên nhỏ nhất > avg = 0
-2.5 -> số nguyên nhỏ nhất > avg = -2


### 3.2. So sánh số thực có rủi ro, hãy dùng số nguyên

`x > tổng / n` tương đương `x * n > tổng` (vì `n > 0`). Cách này **tránh sai số thực**. Và `tổng // n` (chia lấy phần nguyên **làm tròn xuống**) chính là `floor(avg)` với cả số âm: `-5 // 2 = -3`, đúng với `floor(-2.5) = -3`.

In [3]:
total, n = -5, 2
print(total // n, math.floor(total / n))
print(4 * 2 > 8, 5 * 2 > 8)       # x=4: 4 > 4 sai ; x=5: 5 > 4 đúng (avg = 8/2)

-3 -3
False True


### 3.3. `int(x)` cắt về `0`, còn `//` làm tròn xuống

`int(-2.5)` cho `-2` (cắt phần thập phân, **về 0**), còn `-5 // 2` cho `-3` (làm tròn **xuống**). Với số dương hai cách giống nhau, với số âm thì **khác**. Đây là bẫy kinh điển khi trung bình âm.

In [4]:
print(int(-2.5), -5 // 2, int(2.5), 5 // 2)
print(int(-2.5) + 1, "<- sai (<= 0)", -5 // 2 + 1, "<- đúng, rồi lấy max với 1 để thành số dương")

-2 -3 2 2
-1 <- sai (<= 0) -2 <- đúng, rồi lấy max với 1 để thành số dương


### 3.4. `set` để kiểm tra có mặt

`set(nums)` loại trùng và cho phép `x in s` trong `O(1)` trung bình. Ta cần biết "số dương `x` có trong `nums` không", nên chuyển `nums` thành `set` **một lần**.

In [5]:
s = set([3, 5, 5, -1])
print(s, 5 in s, 6 in s)

{3, 5, -1} True False


### 3.5. Vòng `while` tìm "số nhỏ nhất thỏa điều kiện"

Mẫu: đặt ứng viên `x` ở điểm bắt đầu, rồi `while x bị loại: x += 1`. Vì chỉ có `<= 100` giá trị trong `nums`, nên trong tối đa `~101` lần tăng ta chắc chắn gặp số vắng mặt. Vòng lặp này **luôn dừng**.

In [6]:
present = {3, 5}
x = 5            # điểm bắt đầu
while x in present:
    x += 1
print(x)

6


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay `nums = [3, 5]`:

```
tổng = 8, n = 2, trung bình = 4
số nguyên dương > 4: 5, 6, 7, ...
5 có trong nums -> loại ; 6 không có -> đáp án 6
```

Làm tay `nums = [-1, 1, 2]`: trung bình `0.667`. Các số dương `> 0.667`: `1, 2, 3, ...`; `1, 2` có mặt, `3` vắng -> `3`.

Làm tay một ca trung bình **âm**, `nums = [-3, -2]`: trung bình `-2.5`. "Số dương" nghĩa là bắt đầu từ `1`, nhưng `1 > -2.5` đúng, và `1` vắng mặt -> đáp án `1`. Nên **điểm bắt đầu** là `max(1, floor(avg) + 1)`.

Tóm lại: điểm bắt đầu `start = max(1, floor(avg) + 1)`, rồi **bước qua các số đã có trong `nums`** cho tới khi gặp số vắng mặt.

## 5. Cách 1 — Brute Force / baseline

Thử `x = 1, 2, 3, ...` theo thứ tự; số đầu tiên thỏa **cả hai** (`x * n > tổng` và `x not in nums`) là đáp án. Dùng `list` để kiểm tra `x in nums` (tuần tự).

Nút thắt: mỗi lần kiểm tra `x in nums` tốn `O(n)`, và ta thử tới ~`101` ứng viên nên `O(n * 101)`. Ngoài ra ta thử cả các số **không thể** thỏa (nhỏ hơn trung bình) chỉ để loại.

In [7]:
class SolutionBrute:
    def smallestAbsent(self, nums):
        total, n = sum(nums), len(nums)
        x = 1
        while True:
            if x * n > total and x not in nums:        # x > avg (dạng số nguyên) và vắng mặt
                return x
            x += 1

print(SolutionBrute().smallestAbsent([3, 5]))
print(SolutionBrute().smallestAbsent([-1, 1, 2]))
print(SolutionBrute().smallestAbsent([4, -1]))

6
3
2


**Độ phức tạp:** `n = len(nums)`, `V = 100` là biên giá trị. Time `O(n * V)` (tối đa ~`101` ứng viên, mỗi lần tìm `O(n)`), Space `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá 1 — nhảy thẳng đến điểm bắt đầu:** mọi `x <= avg` đều bị loại, nên ta **không cần thử** chúng. Số nguyên nhỏ nhất thỏa `x > avg` là `floor(avg) + 1 = tổng // n + 1`; vì phải dương, `start = max(1, tổng // n + 1)`.

**Bước đột phá 2 — kiểm tra thuộc bằng `set`:** từ `start`, cứ **nếu** `x` có trong `nums` **thì** `x += 1`. Dừng ở số đầu tiên vắng mặt.

**Quy tắc "nếu… thì…":**

- Nếu trung bình `< 0` -> `start = 1` (`max` kéo lên).
- Nếu `x` đã có trong `nums` -> tăng `x`.
- Số vắng mặt đầu tiên từ `start` trở lên là đáp án (vì mọi số trước đó hoặc `<= avg` hoặc có mặt).

Chặn trên: vì `nums[i] <= 100`, trung bình `<= 100`; số `101 > avg` luôn vắng mặt, nên đáp án **`<= 101`**.

In [8]:
def smallest_absent(nums):
    start = max(1, sum(nums) // len(nums) + 1)
    present = set(nums)
    x = start
    while x in present:
        x += 1
    return x

print(smallest_absent([3, 5]), smallest_absent([-1, 1, 2]), smallest_absent([4, -1]), smallest_absent([-3, -2]))

6 3 2 1


## 7. Các cách tiếp theo

### Cách 2 — `set` + điểm bắt đầu (cách nộp bài)

Đúng ý tưởng ở mục 6: tính `start` bằng số nguyên, đưa `nums` vào `set`, rồi tăng cho đến khi vắng mặt.

In [9]:
class SolutionSet:
    def smallestAbsent(self, nums):
        present = set(nums)
        x = max(1, sum(nums) // len(nums) + 1)
        while x in present:
            x += 1
        return x

print(SolutionSet().smallestAbsent([3, 5]), SolutionSet().smallestAbsent([-1, 1, 2]), SolutionSet().smallestAbsent([4, -1]))

6 3 2


**Độ phức tạp:** Time `O(n + V)` (dựng `set` rồi tối đa ~`V` bước tăng), Space `O(n)`.

### Cách 3 — Mảng cờ kích thước cố định

Giá trị dương nhỏ nhất cần xét chỉ nằm trong `1..101`. Dùng `seen = [False] * 102` rồi đánh dấu `seen[v] = True` cho mọi `v` trong `nums` mà `v > 0`. Không cần `set`, không băm. Mẫu "counting array" cho miền giá trị nhỏ.

In [10]:
class SolutionFlags:
    def smallestAbsent(self, nums):
        seen = [False] * 102                     # chỉ số 0..101
        for v in nums:
            if v > 0:
                seen[v] = True
        x = max(1, sum(nums) // len(nums) + 1)
        while seen[x]:
            x += 1
        return x

print(SolutionFlags().smallestAbsent([3, 5]), SolutionFlags().smallestAbsent([-1, 1, 2]), SolutionFlags().smallestAbsent([4, -1]))

6 3 2


**Độ phức tạp:** Time `O(n + V)`, Space `O(V)` (mảng `102` ô). `seen[x]` không bao giờ vượt chỉ số vì `x <= 101`.

### Cách 4 — Sắp xếp rồi quét

Sắp các giá trị khác nhau tăng dần; đặt ứng viên `cand = start`. Duyệt từng `v`: nếu `v < cand` bỏ qua, nếu `v == cand` thì `cand += 1`, nếu `v > cand` thì dừng (khoảng trống). Cách này là "merge hai dãy tăng" quen thuộc.

In [11]:
class SolutionSorted:
    def smallestAbsent(self, nums):
        cand = max(1, sum(nums) // len(nums) + 1)
        for v in sorted(set(nums)):
            if v < cand:
                continue
            if v == cand:
                cand += 1
            else:
                break
        return cand

print(SolutionSorted().smallestAbsent([3, 5]), SolutionSorted().smallestAbsent([-1, 1, 2]), SolutionSorted().smallestAbsent([4, -1]))

6 3 2


**Độ phức tạp:** Time `O(n log n)` (sắp xếp), Space `O(n)`.

## 8. TRACE từng bước

Trace `nums = [3, 5]`: `tổng = 8`, `n = 2`, `8 // 2 + 1 = 5`, `start = max(1, 5) = 5`, `present = {3, 5}`.

| Bước | x | `x in present`? | hành động |
|---|---|---|---|
| 0 | 5 | có | `x = 6` |
| 1 | 6 | không | dừng, đáp án `6` |

Trace `nums = [-3, -2]`: `tổng = -5`, `n = 2`, `-5 // 2 + 1 = -2`, `start = max(1, -2) = 1`; `1` vắng mặt -> đáp án `1`.

In [12]:
def trace_absent(nums):
    total, n = sum(nums), len(nums)
    floor_avg = total // n
    start = max(1, floor_avg + 1)
    print(f"nums={nums}: tổng={total}, n={n}, tổng//n={floor_avg}, start=max(1, {floor_avg + 1})={start}")
    present = set(nums)
    x = start
    step = 0
    while x in present:
        print(f"  bước {step}: x={x} có trong nums -> x += 1")
        x += 1
        step += 1
    print(f"  bước {step}: x={x} vắng mặt -> đáp án {x}")
    return x

trace_absent([3, 5])
trace_absent([-1, 1, 2])
trace_absent([-3, -2])
trace_absent([100] * 5)

nums=[3, 5]: tổng=8, n=2, tổng//n=4, start=max(1, 5)=5
  bước 0: x=5 có trong nums -> x += 1
  bước 1: x=6 vắng mặt -> đáp án 6
nums=[-1, 1, 2]: tổng=2, n=3, tổng//n=0, start=max(1, 1)=1
  bước 0: x=1 có trong nums -> x += 1
  bước 1: x=2 có trong nums -> x += 1
  bước 2: x=3 vắng mặt -> đáp án 3
nums=[-3, -2]: tổng=-5, n=2, tổng//n=-3, start=max(1, -2)=1
  bước 0: x=1 vắng mặt -> đáp án 1
nums=[100, 100, 100, 100, 100]: tổng=500, n=5, tổng//n=100, start=max(1, 101)=101
  bước 0: x=101 vắng mặt -> đáp án 101


101

## 9. Code nộp LeetCode

In [13]:
from typing import List, Optional

class Solution:
    def smallestAbsent(self, nums: List[int]) -> int:
        present = set(nums)
        x = max(1, sum(nums) // len(nums) + 1)   # số nguyên dương nhỏ nhất > trung bình
        while x in present:
            x += 1
        return x

## 10. Vì sao thuật toán đúng

- **Điểm bắt đầu đúng:** `tổng // n = floor(avg)`, nên `tổng // n + 1` là số nguyên **nhỏ nhất lớn hơn hẳn** `avg` (kể cả khi `avg` nguyên). Lấy `max` với `1` để đảm bảo **dương**. Vậy mọi `x < start` hoặc `x <= avg` hoặc `x <= 0`, bị loại đúng.
- **Dừng đúng chỗ:** ta tăng `x` từ `start` và chỉ tăng khi `x` **có mặt** trong `nums`. Số đầu tiên vắng mặt là số nhỏ nhất `>= start` không có trong `nums`, tức thỏa cả hai điều kiện và nhỏ nhất.
- **Vòng lặp luôn dừng:** `nums` chỉ có `<= 100` phần tử, mỗi phần tử `<= 100`, nên `101` (hoặc sớm hơn) chắc chắn vắng mặt. Đáp án luôn `<= 101`.
- **Tránh sai số thực:** `//` là phép chia nguyên làm tròn xuống, không có sai số; số âm cũng đúng vì Python làm tròn về `-vô cực`.

## 11. Phân tích Time / Space Complexity

`n = len(nums) <= 100`, `V = 100` (giá trị tối đa).

| Cách | Time | Space |
|---|---|---|
| Thử từng `x`, kiểm tra trong `list` | `O(n * V)` | `O(1)` |
| `set` + điểm bắt đầu | `O(n + V)` | `O(n)` |
| Mảng cờ `102` ô | `O(n + V)` | `O(V)` |
| Sắp xếp rồi quét | `O(n log n)` | `O(n)` |

Tất cả đều rất nhanh với `n <= 100`. Cách `set` ngắn gọn nhất và độc lập với cận `V`.

## 12. Bộ test

Ví dụ của đề, các biên (mọi phần tử `100`, mọi phần tử `-100`, một phần tử) và đối chiếu **tất cả cách**.

In [14]:
impls = [SolutionBrute(), SolutionSet(), SolutionFlags(), SolutionSorted(), Solution()]

cases = [
    ([3, 5], 6), ([-1, 1, 2], 3), ([4, -1], 2),
    ([1], 2),                        # trung bình 1 -> phải > 1 -> 2
    ([2], 3),
    ([-5], 1),                       # trung bình âm -> 1
    ([100] * 100, 101),              # trung bình 100 -> 101
    ([-100] * 100, 1),
    ([0], 1),
    ([1, 2, 3, 4], 5),               # trung bình 2.5 -> 3 và 4 có mặt -> 5
]
for nums, exp in cases:
    for impl in impls:
        got = impl.smallestAbsent(nums[:])
        assert got == exp, (type(impl).__name__, nums[:5], got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 10 ca


In [15]:
import random
random.seed(3678)

def ref(nums):
    n, total = len(nums), sum(nums)
    x = 1
    while not (x * n > total and x not in nums):
        x += 1
    return x

for _ in range(3000):
    n = random.randint(1, 12)
    lo = random.choice([-100, -10, 1])
    hi = random.choice([10, 100])
    nums = [random.randint(lo, hi) for _ in range(n)]
    exp = ref(nums)
    for impl in impls:
        assert impl.smallestAbsent(nums[:]) == exp, (nums, type(impl).__name__)
    assert 1 <= exp <= 101
print("3000 test ngẫu nhiên (seed=3678) OK")

3000 test ngẫu nhiên (seed=3678) OK


In [16]:
# Mảng dài tới 100 phần tử và nhiều giá trị lặp; kiểm tra thêm tính chất của đáp án
for _ in range(500):
    n = random.randint(1, 100)
    nums = [random.randint(-100, 100) for _ in range(n)]
    ans = Solution().smallestAbsent(nums)
    assert ans > 0 and ans not in nums and ans * n > sum(nums)          # thỏa đề
    for smaller in range(1, ans):                                       # và là nhỏ nhất
        assert smaller in nums or smaller * n <= sum(nums)
print("500 mảng dài: đáp án thỏa đề và nhỏ nhất")

500 mảng dài: đáp án thỏa đề và nhỏ nhất


## 13. Các lỗi người mới hay gặp

### Lỗi 1: dùng `>=` thay cho `>`

Đề yêu cầu **lớn hơn hẳn** trung bình. Với `[3,5]` trung bình `4`, nên `4` không được tính.

In [17]:
def wrong_ge(nums):
    present = set(nums)
    x = max(1, -(-sum(nums) // len(nums)))        # ceil(avg): số nguyên nhỏ nhất >= avg  (SAI: phải > avg)
    while x in present:
        x += 1
    return x

print(wrong_ge([3, 5]), "!=", Solution().smallestAbsent([3, 5]))
assert wrong_ge([3, 5]) != Solution().smallestAbsent([3, 5])        # 4 != 6

4 != 6


### Lỗi 2: bỏ qua điều kiện trung bình

Chỉ tìm "số dương nhỏ nhất vắng mặt" (bài First Missing Positive) là sai đề.

In [18]:
def wrong_ignore_avg(nums):
    present = set(nums)
    x = 1
    while x in present:
        x += 1
    return x

print(wrong_ignore_avg([3, 5]), "!=", Solution().smallestAbsent([3, 5]))
assert wrong_ignore_avg([3, 5]) != Solution().smallestAbsent([3, 5])   # 1 != 6

1 != 6


### Lỗi 3: dùng `int()` cho trung bình âm và quên ép dương

`int(-2.5) + 1 = -1`, không phải số dương. Kết quả trả về `-1` (hoặc `0`) vi phạm điều kiện "dương".

In [19]:
def wrong_int_trunc(nums):
    present = set(nums)
    x = int(sum(nums) / len(nums)) + 1            # SAI với avg âm: không max với 1
    while x in present:
        x += 1
    return x

print(wrong_int_trunc([-3, -2]), "!=", Solution().smallestAbsent([-3, -2]))
assert wrong_int_trunc([-3, -2]) != Solution().smallestAbsent([-3, -2])   # -1 != 1

-1 != 1


### Lỗi 4: quên bước qua các số đã có trong `nums`

Trả thẳng `floor(avg) + 1` mà không kiểm tra số đó có trong `nums` không. Với `[3,5]` điểm bắt đầu là `5`, nhưng `5` đang có mặt.

In [20]:
def wrong_no_skip(nums):
    return max(1, sum(nums) // len(nums) + 1)

print(wrong_no_skip([3, 5]), "!=", Solution().smallestAbsent([3, 5]))
assert wrong_no_skip([3, 5]) != Solution().smallestAbsent([3, 5])         # 5 != 6

5 != 6


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Số nguyên dương nhỏ nhất **không có** trong mảng" -> `set` + vòng `while x in s: x += 1` (First Missing Positive, phiên bản đơn giản).
- Điều kiện "**lớn hơn** một giá trị thực" -> chuyển thành **điểm bắt đầu** `floor(v) + 1` thay vì thử từng số.
- Với trung bình luôn nghĩ **số nguyên hóa**: so sánh `x * n > tổng` hoặc dùng `tổng // n`.
- Cẩn thận `int()` (cắt về `0`) và `//` (làm tròn xuống): chỉ khác nhau với số **âm**.
- "Dương" nghĩa là `>= 1`: nhớ `max(1, ...)`.
- Miền giá trị nhỏ (`|a[i]| <= 100`) -> có thể dùng **mảng cờ** thay `set` và cho chặn trên của đáp án (`<= 101`).

## 15. Biến thể của bài

1. **First Missing Positive** (LeetCode 41, Hard): tìm số dương nhỏ nhất vắng mặt trong `O(n)` thời gian, `O(1)` bộ nhớ phụ.
2. **Missing Number** (268): thiếu một số trong `0..n`.
3. Số nguyên **nhỏ nhất lớn hơn trung vị** (thay vì trung bình) và vắng mặt.
4. Số vắng mặt **lớn nhất không vượt quá trung bình**.
5. Trả về **tất cả** số dương vắng mặt trong khoảng `(avg, max(nums)]`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Thử từng `x` + `list` | `x * n > tổng and x not in nums` | `O(n V)` | `O(1)` | Làm đúng đề, viết test đối chiếu |
| `set` + `start` | nhảy đến `floor(avg) + 1`, bước qua số có mặt | `O(n + V)` | `O(n)` | Cách nộp chuẩn |
| Mảng cờ | `seen[v]` với `v` dương | `O(n + V)` | `O(V)` | Khi miền giá trị nhỏ và biết trước |
| Sắp xếp + quét | `cand` đuổi theo giá trị đã sắp | `O(n log n)` | `O(n)` | Khi muốn xử lý dạng "lấp khoảng trống" |

Chọn `set` + `start` để nộp; nó ngắn và không lệ thuộc cận `V`.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().smallestAbsent([1, 2, 3])` là gì?

**Bài 2 (tính tay):** `nums = [-4, 0, 7]`: tính trung bình, điểm bắt đầu và đáp án.

**Bài 3 (điền TODO):** hoàn thành `start_candidate(nums)` trả `max(1, tổng // n + 1)`.

**Bài 4 (tự viết lại):** viết `smallest_absent_ge(nums)` trả số dương nhỏ nhất vắng mặt và **`>=` trung bình** (biến thể dùng `>=`).

**Bài 5 (mini challenge):** viết `k_smallest_absent(nums, k)` trả `k` số dương nhỏ nhất vắng mặt và lớn hơn hẳn trung bình (tăng dần).

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().smallestAbsent([1, 2, 3]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def start_candidate_todo(nums):
    # TODO: trả về số nguyên dương nhỏ nhất > trung bình
    return 0

4


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: trung bình `2`, số `> 2` vắng mặt. Bài 2: tổng `3`, `n = 3`.

**Mức 2 (vừa):** Bài 3: `max(1, sum(nums) // len(nums) + 1)`. Bài 4: điểm bắt đầu là số nguyên nhỏ nhất `>= avg`, tức `-(-tổng // n)` (làm tròn lên) rồi `max(1, ...)`. Bài 5: lặp lại vòng tăng `x` và thu thập `k` số vắng mặt.

**Mức 3 (gần đáp án):** Bài 1: `4`. Bài 2: trung bình `1`, nên phải `> 1`: ứng viên `2` vắng mặt -> `2`. Bài 4: giống Solution nhưng điểm bắt đầu là `ceil(avg)` chứ không phải `floor(avg) + 1`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: trung bình 2 -> 3 có mặt -> 4
assert Solution().smallestAbsent([1, 2, 3]) == 4

# Bài 2: [-4, 0, 7] tổng 3, n 3, trung bình 1 -> start = max(1, 3//3+1) = 2 -> 2 vắng mặt
assert sum([-4, 0, 7]) // 3 + 1 == 2
assert Solution().smallestAbsent([-4, 0, 7]) == 2

# Bài 3
def start_candidate(nums):
    return max(1, sum(nums) // len(nums) + 1)
assert start_candidate([3, 5]) == 5 and start_candidate([-3, -2]) == 1

# Bài 4
def smallest_absent_ge(nums):
    present = set(nums)
    x = max(1, -(-sum(nums) // len(nums)))            # ceil(avg), tối thiểu 1
    while x in present:
        x += 1
    return x
assert smallest_absent_ge([3, 5]) == 4                 # >= 4 và vắng mặt
assert smallest_absent_ge([1, 2, 3]) == 4
assert smallest_absent_ge([-1, 1, 2]) == 3

# Bài 5
def k_smallest_absent(nums, k):
    present = set(nums)
    x = start_candidate(nums)
    out = []
    while len(out) < k:
        if x not in present:
            out.append(x)
        x += 1
    return out
assert k_smallest_absent([3, 5], 3) == [6, 7, 8]
assert k_smallest_absent([4, -1], 2) == [2, 3]
assert k_smallest_absent([1, 2, 3], 1) == [Solution().smallestAbsent([1, 2, 3])]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "số dương nhỏ nhất vắng mặt, lớn hơn hẳn một ngưỡng":**

```python
present = set(nums)
x = max(1, sum(nums) // len(nums) + 1)   # nhảy tới số nguyên nhỏ nhất > trung bình, ép dương
while x in present:                      # bước qua các số đã có
    x += 1
return x
```

**Checklist trước khi submit:**

- [ ] Điều kiện là `>` ngặt (không `>=`) so với trung bình?
- [ ] Dùng `//` (làm tròn xuống), không `int()` khi trung bình có thể âm?
- [ ] `max(1, ...)` để kết quả luôn **dương**?
- [ ] Bước qua các số **có mặt** trong `nums` bằng `set`?
- [ ] Đã thử `[3,5]`, `[-1,1,2]`, `[4,-1]`, `[100]*100`, `[-100]*100`, `[1]`?
- [ ] Chữ ký đúng `smallestAbsent(self, nums: List[int]) -> int`?